In [1]:
# Data preprocessing: cleaning, z-score normalization, and one-hot encoding
# Re-importing libraries
import sys
import subprocess
import warnings
warnings.filterwarnings('ignore')

def ensure_package(pkg_name, import_name=None):
    """Install package if import fails."""
    import importlib
    try:
        importlib.import_module(import_name or pkg_name)
    except Exception:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg_name])
        except Exception as e:
            print(f'Package install skipped/failed for {pkg_name}: {e}')

for pkg, imp in [
    ('pandas', 'pandas'),
    ('numpy', 'numpy'),
    ('matplotlib', 'matplotlib'),
    ('scikit-learn', 'sklearn'),
    ('seaborn', 'seaborn'),
    ('xgboost', 'xgboost'),
    ('shap', 'shap'),
]:
    ensure_package(pkg, imp)

import os
import re
import json
import math
import textwrap
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.base import clone
from xgboost import XGBClassifier
import shap

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')
np.random.seed(42)
random.seed(42)


import pandas as pd
file_name = 'Foodpanda Analysis Dataset.csv'
print('\n' + '='*80)
print('STEP 2: UPLOAD CSV FILE')
print('='*80)
df = pd.read_csv(file_name)
print(f'Loaded file: {file_name}')
print(f'Dataset shape: {df.shape}')
display(df.head())


def safe_to_datetime(series):
    return pd.to_datetime(series, errors='coerce')

def cap_outliers_iqr(df_numeric, numeric_features_list):
    """Caps outliers in numeric columns using the IQR method."""
    capping_log = []
    df_capped = df_numeric.copy()
    for col in numeric_features_list:
        if pd.api.types.is_numeric_dtype(df_capped[col]):
            Q1 = df_capped[col].quantile(0.25)
            Q3 = df_capped[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR

            num_outliers_lower = (df_capped[col] < lower_bound).sum()
            num_outliers_upper = (df_capped[col] > upper_bound).sum()

            df_capped[col] = np.where(df_capped[col] < lower_bound, lower_bound, df_capped[col])
            df_capped[col] = np.where(df_capped[col] > upper_bound, upper_bound, df_capped[col])

            if num_outliers_lower > 0 or num_outliers_upper > 0:
                capping_log.append({
                    'variable': col,
                    'lower_bound_cap': lower_bound,
                    'upper_bound_cap': upper_bound,
                    'outliers_capped_lower': num_outliers_lower,
                    'outliers_capped_upper': num_outliers_upper
                })
    return df_capped, pd.DataFrame(capping_log)

def get_feature_names_from_preprocessor(preprocessor, numeric_features, categorical_features):
    """Extracts feature names after preprocessing."""
    feature_names = []
    cat_feature_names = []

    # Get names for numeric features (scaled)
    feature_names.extend(numeric_features)

    # Get names for one-hot encoded categorical features
    for name, transformer, features in preprocessor.transformers_:
        if name == 'cat': # Assuming 'cat' is the name given to the categorical pipeline
            onehot_encoder = transformer.named_steps['onehot']
            # This method works for scikit-learn >= 0.23
            if hasattr(onehot_encoder, 'get_feature_names_out'):
                # Get feature names from the OneHotEncoder
                encoded_feature_names = onehot_encoder.get_feature_names_out(categorical_features)
                feature_names.extend(encoded_feature_names)
                cat_feature_names.extend(encoded_feature_names)
            else:
                # Fallback for older scikit-learn versions or if get_feature_names_out is not available
                for i, col in enumerate(categorical_features):
                    for category in onehot_encoder.categories_[i]:
                        encoded_name = f'{col}_{category}'
                        feature_names.append(encoded_name)
                        cat_feature_names.append(encoded_name)
    return feature_names, cat_feature_names

#
print('STEP 3: DATA PREPROCESSING')

df_clean = df.copy()

# Drop unnamed or all-missing columns
cols_to_drop = [c for c in df_clean.columns if c.lower().startswith('unnamed') or df_clean[c].isna().all()]
if cols_to_drop:
    print(f'Dropping empty/useless columns: {cols_to_drop}')
    df_clean = df_clean.drop(columns=cols_to_drop)

# Create binary target for cancellation
if 'delivery_status' not in df_clean.columns:
    raise ValueError("The target column 'delivery_status' is missing.")

df_clean['target_cancelled'] = (df_clean['delivery_status'].astype(str).str.strip().str.lower() == 'cancelled').astype(int)
print('Target distribution (0 = not cancelled, 1 = cancelled):')
print(df_clean['target_cancelled'].value_counts())

# Parse dates safely
for dc in [c for c in df_clean.columns if 'date' in c.lower()]:
    df_clean[dc] = safe_to_datetime(df_clean[dc])

# Feature engineering from dates
if 'order_date' in df_clean.columns:
    df_clean['order_year'] = df_clean['order_date'].dt.year
    df_clean['order_month'] = df_clean['order_date'].dt.month
    df_clean['order_dayofweek'] = df_clean['order_date'].dt.dayofweek
    df_clean['order_day'] = df_clean['order_date'].dt.day
    df_clean['order_weekofyear'] = df_clean['order_date'].dt.isocalendar().week.astype('Int64')

if 'signup_date' in df_clean.columns and 'order_date' in df_clean.columns:
    df_clean['days_since_signup'] = (df_clean['order_date'] - df_clean['signup_date']).dt.days
    df_clean.loc[df_clean['days_since_signup'] < 0, 'days_since_signup'] = np.nan

if 'last_order_date' in df_clean.columns and 'order_date' in df_clean.columns:
    df_clean['days_since_last_order'] = (df_clean['order_date'] - df_clean['last_order_date']).dt.days
    df_clean.loc[df_clean['days_since_last_order'] < 0, 'days_since_last_order'] = np.nan

# Columns to exclude from modeling because they are IDs, leakage, or raw target
exclusion_cols = [
    'customer_id', 'order_id', 'delivery_status',
    'rating', 'rating_date', 'churned',
    'signup_date', 'order_date', 'last_order_date'
]
exclusion_cols = [c for c in exclusion_cols if c in df_clean.columns]
print(f'Columns excluded from modeling: {exclusion_cols}')

model_df = df_clean.drop(columns=exclusion_cols, errors='ignore').copy()

# Separate target and features
X = model_df.drop(columns=['target_cancelled'])
y = model_df['target_cancelled']

# Identify current column types
numeric_features = X.select_dtypes(include=[np.number, 'Int64']).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]
print(f'Numeric features: {numeric_features}')
print(f'Categorical features: {categorical_features}')

# Cap numeric outliers before train-test split for easier reproducibility in EDA.
X_before_cap = X.copy()
X[numeric_features], capping_log = cap_outliers_iqr(X[numeric_features], numeric_features)
print('\nOutlier capping summary:')
display(capping_log)

# Split data: 50% training, 50% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.50, random_state=42, stratify=y
)
print(f'Training shape: {X_train.shape}, Testing shape: {X_test.shape}')

# Preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())  # z-score normalization
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Fit preprocessing and generate encoded tables
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names, cat_feature_names = get_feature_names_from_preprocessor(preprocessor, numeric_features, categorical_features)

X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

print('\nProcessed training data preview:')
display(X_train_processed_df.head())

# Create One-Hot-Encoding mapping table
encoding_rows = []
for encoded_name in cat_feature_names:
    matched = False
    for original_col in categorical_features:
        prefix = original_col + '_'
        if encoded_name.startswith(prefix):
            category_value = encoded_name[len(prefix):]
            encoding_rows.append({
                'original_variable': original_col,
                'category_value': category_value,
                'encoded_column': encoded_name,
                'encoded_meaning': f"1 = {original_col} is '{category_value}', 0 = otherwise"
            })
            matched = True
            break
    if not matched:
        encoding_rows.append({
            'original_variable': 'unknown',
            'category_value': 'unknown',
            'encoded_column': encoded_name,
            'encoded_meaning': '1 indicates this encoded category is active'
        })

ohe_mapping_df = pd.DataFrame(encoding_rows)
print('\nOne-Hot-Encoding mapping table:')
display(ohe_mapping_df)


STEP 2: UPLOAD CSV FILE
Loaded file: Foodpanda Analysis Dataset.csv
Dataset shape: (6000, 21)


,customer_id,gender,age,city,signup_date,order_id,order_date,restaurant_name,dish_name,category,...,price,payment_method,order_frequency,last_order_date,loyalty_points,churned,rating,rating_date,delivery_status,Unnamed: 20
0,C5221,Male,Senior,Lahore,10/3/2023,O9221,1/10/2024,McDonald's,Burger,Italian,...,1291.14,Card,7,8/21/2025,42,Inactive,3,11/29/2024,Cancelled,NaN
1,C2831,Male,Adult,Multan,7/7/2024,O6831,8/23/2023,KFC,Burger,Italian,...,956.04,Wallet,24,11/25/2024,81,Active,2,8/21/2025,Delayed,NaN
2,C2851,Other,Senior,Multan,6/20/2025,O6851,8/23/2023,Pizza Hut,Fries,Italian,...,882.51,Cash,42,5/10/2025,82,Inactive,3,9/19/2024,Delayed,NaN
3,C1694,Female,Senior,Peshawar,9/5/2023,O5694,8/23/2023,Subway,Pizza,Dessert,...,231.30,Card,27,7/24/2025,45,Inactive,2,6/29/2025,Delayed,NaN
4,C5641,Other,Senior,Islamabad,10/13/2023,O9641,3/27/2024,Pizza Hut,Burger,Chinese,...,620.96,Card,17,8/21/2025,206,Active,2,4/9/2025,Cancelled,NaN


STEP 3: DATA PREPROCESSING
Dropping empty/useless columns: ['Unnamed: 20']
Target distribution (0 = not cancelled, 1 = cancelled):
target_cancelled
0    4032
1    1968
Name: count, dtype: int64
Columns excluded from modeling: ['customer_id', 'order_id', 'delivery_status', 'rating', 'rating_date', 'churned', 'signup_date', 'order_date', 'last_order_date']
Numeric features: ['quantity', 'price', 'order_frequency', 'loyalty_points', 'order_year', 'order_month', 'order_dayofweek', 'order_day', 'order_weekofyear', 'days_since_signup', 'days_since_last_order']
Categorical features: ['gender', 'age', 'city', 'restaurant_name', 'dish_name', 'category', 'payment_method']

Outlier capping summary:


""


Training shape: (3000, 18), Testing shape: (3000, 18)

Processed training data preview:


,quantity,price,order_frequency,loyalty_points,order_year,order_month,order_dayofweek,order_day,order_weekofyear,days_since_signup,...,dish_name_Pizza,dish_name_Sandwich,category_Chinese,category_Continental,category_Dessert,category_Fast Food,category_Italian,payment_method_Card,payment_method_Cash,payment_method_Wallet
5269,-0.688456,-0.268878,-0.514977,1.147774,1.262860,-0.454022,0.481855,0.846274,-0.383550,-0.204425,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
2394,-0.688456,-0.756400,0.389631,-0.139857,-0.187037,1.585434,0.481855,1.304359,1.697920,1.118410,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2782,1.443721,0.423980,1.711749,-0.965621,-0.187037,0.128680,1.476395,1.418881,0.220748,-0.131385,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4088,-0.688456,-0.641248,-1.071658,-1.490471,-0.187037,-0.454022,-0.512685,-0.069896,-0.450694,0.314970,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3791,0.022269,-1.374067,-0.793317,1.504672,-0.187037,1.294084,0.979125,0.846274,1.362199,-0.131385,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0



One-Hot-Encoding mapping table:


,original_variable,category_value,encoded_column,encoded_meaning
0,gender,Female,gender_Female,"1 = gender is 'Female', 0 = otherwise"
1,gender,Male,gender_Male,"1 = gender is 'Male', 0 = otherwise"
2,gender,Other,gender_Other,"1 = gender is 'Other', 0 = otherwise"
3,age,Adult,age_Adult,"1 = age is 'Adult', 0 = otherwise"
4,age,Senior,age_Senior,"1 = age is 'Senior', 0 = otherwise"
5,age,Teenager,age_Teenager,"1 = age is 'Teenager', 0 = otherwise"
6,city,Islamabad,city_Islamabad,"1 = city is 'Islamabad', 0 = otherwise"
7,city,Karachi,city_Karachi,"1 = city is 'Karachi', 0 = otherwise"
8,city,Lahore,city_Lahore,"1 = city is 'Lahore', 0 = otherwise"
9,city,Multan,city_Multan,"1 = city is 'Multan', 0 = otherwise"
